# 🛡️ Secure Enterprise AI Copilot

A starter GenAI portfolio notebook demonstrating role-based document access, permission-aware retrieval, source-grounded excerpts, and minimal audit metadata.

**Important:** This is an educational prototype, not a production security boundary. The documents and roles are synthetic examples.

## 1. Install dependencies

In [ ]:
%pip install -q scikit-learn pandas

## 2. Imports

In [ ]:
from datetime import datetime, timezone
from typing import Dict, List, Any
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
print('Environment ready.')

## 3. Synthetic enterprise documents
Each document declares which demo roles may access it.

In [ ]:
documents = [
    {"id": "HR-001", "title": "Annual Leave Policy",
     "text": "Employees submit annual leave requests through the HR portal. Managers approve requests based on team coverage and company policy.",
     "allowed_roles": ["employee", "manager", "hr"]},
    {"id": "IT-001", "title": "Password and MFA Guidance",
     "text": "Use a unique password and enable multi-factor authentication. Never share passwords or one-time codes. Report suspected account compromise to IT security.",
     "allowed_roles": ["employee", "manager", "it", "security", "hr"]},
    {"id": "FIN-001", "title": "Expense Reimbursement",
     "text": "Submit eligible business expenses with receipts through the finance portal. Finance reviews claims according to policy.",
     "allowed_roles": ["employee", "manager", "finance"]},
    {"id": "SEC-001", "title": "Incident Response Procedure",
     "text": "Report suspected security incidents to the security team immediately. Preserve evidence and do not investigate beyond your authorization.",
     "allowed_roles": ["security", "it", "manager"]},
    {"id": "HR-002", "title": "Confidential Compensation Guidance",
     "text": "Compensation planning records are restricted to authorized HR personnel and designated leadership.",
     "allowed_roles": ["hr", "leadership"]},
]
pd.DataFrame([{"id": d["id"], "title": d["title"], "allowed_roles": ", ".join(d["allowed_roles"])} for d in documents])

## 4. Access-control filter
In production, roles must come from a trusted identity provider and be enforced server-side—not from user-editable input.

In [ ]:
def authorized_documents(user_role: str, docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    role = user_role.strip().lower()
    return [doc for doc in docs if role in doc["allowed_roles"]]

authorized_documents("employee", documents)

## 5. Retrieve only authorized documents
TF-IDF is used as a transparent baseline. Similarity scores help rank matches; they do not guarantee a correct answer.

In [ ]:
def retrieve_authorized(query: str, user_role: str, top_k: int = 3) -> List[Dict[str, Any]]:
    if not query or not query.strip():
        raise ValueError("Query cannot be empty.")
    allowed = authorized_documents(user_role, documents)
    if not allowed:
        return []
    corpus = [d["title"] + ". " + d["text"] for d in allowed]
    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(corpus + [query])
    scores = cosine_similarity(matrix[-1], matrix[:-1]).ravel()
    ranked = sorted(zip(allowed, scores), key=lambda x: x[1], reverse=True)
    return [{**doc, "similarity": float(score)} for doc, score in ranked[:max(1, top_k)] if score > 0]

retrieve_authorized("How do I request annual leave?", "employee")

## 6. Produce a source-grounded answer
This demo returns relevant authorized excerpts rather than generating unsupported claims with an LLM.

In [ ]:
def answer_question(query: str, user_role: str, top_k: int = 3) -> Dict[str, Any]:
    hits = retrieve_authorized(query, user_role, top_k)
    if not hits:
        answer = "I couldn't find a relevant answer in the documents available to your role."
    else:
        answer = "\n\n".join(
            f"**{d['title']} ({d['id']})**: {d['text']}" for d in hits
        )
    return {
        "answer": answer,
        "role": user_role,
        "sources": [{"id": d["id"], "title": d["title"], "similarity": round(d["similarity"], 3)} for d in hits]
    }

answer_question("How do I request annual leave?", "employee")

## 7. Minimal audit metadata
Avoid recording full prompts, answers, or confidential document contents by default.

In [ ]:
AUDIT_LOG: List[Dict[str, Any]] = []

def copilot_query(query: str, user_role: str, top_k: int = 3) -> Dict[str, Any]:
    result = answer_question(query, user_role, top_k)
    AUDIT_LOG.append({
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "role": user_role,
        "query_length": len(query),
        "source_ids": [s["id"] for s in result["sources"]],
        "source_count": len(result["sources"])
    })
    return result

copilot_query("What is the MFA guidance?", "employee")

## 8. Basic access-control tests

In [ ]:
assert all(d["id"] != "HR-002" for d in authorized_documents("employee", documents))
assert any(d["id"] == "HR-002" for d in authorized_documents("hr", documents))
try:
    retrieve_authorized("  ", "employee")
except ValueError:
    pass
else:
    raise AssertionError("Empty query should raise ValueError.")
employee_hits = retrieve_authorized("confidential compensation planning", "employee", top_k=5)
assert all(d["id"] != "HR-002" for d in employee_hits)
print("Basic access-control tests passed.")

## 9. Inspect audit log

In [ ]:
pd.DataFrame(AUDIT_LOG)

## 🔐 Production security checklist

- [ ] Authenticate users with a trusted identity provider.
- [ ] Enforce authorization server-side before retrieval and before tool/action execution.
- [ ] Re-check document permissions when access changes.
- [ ] Treat retrieved text as untrusted input and defend against prompt injection.
- [ ] Keep secrets in a secrets manager.
- [ ] Minimize logs and restrict access to audit records.
- [ ] Test for cross-user data leakage and permission bypass.
- [ ] Review model-provider data handling before sending enterprise content.

## 🗺️ Roadmap

- [ ] Connect an approved enterprise document store
- [ ] Add an LLM answer layer with source citations
- [ ] Replace demo roles with identity-provider claims
- [ ] Add permission-aware chunking and retrieval
- [ ] Build a Streamlit interface
- [ ] Add adversarial and regression tests

## 📄 License

Choose a license and add the corresponding `LICENSE` file before publishing.